In this notebook we use the bounding boxes given with the [Imagenet data set](https://www.kaggle.com/c/imagenet-object-localization-challenge) to generate a new sub-set. From the original data set `Imagenet_full` by only keeping the smallest square comprising the bounding boxe we generate `Imagenet_bbox` sub-set




In [1]:
from retinotopy import *
welcome()

Running on GPU :  Tesla V100-SXM2-32GB #GPU= 1
Running on Jean Zay with Tesla V100-SXM2-32GB with DATAROOT='/lustre/fsn1/projects/rech/fsx/uvb28bo/data' and USER='uvb28bo' 


------------------------------------------------------------------------------------------
On date 2025-06-20, Running learning on host r6i6n1 with device cuda, pytorch==2.7.0+cu126
------------------------------------------------------------------------------------------
Welcome on Linux-5.14.0-427.76.1.el9_4.x86_64-x86_64-with-glibc2.34


In [2]:
args = Params()
source_data_set_type = 'full'
args.root  = f'{DATAROOT}/Imagenet_{source_data_set_type}' # Directory containing original images
target_data_set_type = 'bbox'
args.target_root = f'{DATAROOT}/Imagenet_{target_data_set_type}' # Directory containing cropped images
os.makedirs(args.target_root, exist_ok=True)
args.folders, args.root, args.target_root#, revlabels_dico

(['val', 'train'],
 '/lustre/fsn1/projects/rech/fsx/uvb28bo/data/Imagenet_full',
 '/lustre/fsn1/projects/rech/fsx/uvb28bo/data/Imagenet_bbox')

## reading localisation metadata

First for the 'train' dataset:

In [3]:
def get_boxes(df, value):
    idx = list(df['ImageId'][df['ImageId'] == value].index)
    bboxes = []
    if idx:
        for i in range(len(df["PredictionString"][idx[0]].split(' '))//5):
            pos =(5*i)
            bboxes.append({'xmin' : int(df["PredictionString"][idx[0]].split(' ')[1 + pos]),
                           'ymin' : int(df["PredictionString"][idx[0]].split(' ')[2 + pos]),
                           'xmax' : int(df["PredictionString"][idx[0]].split(' ')[3 + (5 *i)]),
                           'ymax' : int(df["PredictionString"][idx[0]].split(' ')[4 + (5 *i)])
                                        })
    return bboxes

In [4]:
with open(args.annotations_train, 'r') as csv_file:
    df_data = pd.read_csv(csv_file)
df_data.head()

,ImageId,PredictionString
0,n02017213_7894,n02017213 115 49 448 294
1,n02017213_7261,n02017213 91 42 330 432
2,n02017213_5636,n02017213 230 104 414 224
3,n02017213_6132,n02017213 46 82 464 387
4,n02017213_7659,n02017213 103 66 331 335


In [5]:
get_boxes(df_data, 'n02099849_2300')

[{'xmin': 151, 'ymin': 146, 'xmax': 332, 'ymax': 333},
 {'xmin': 7, 'ymin': 232, 'xmax': 331, 'ymax': 467}]

In [6]:
get_boxes(df_data, 'n01440764_32420')

[]

Now for the 'val' dataset:

In [7]:
with open(args.annotations_val, 'r') as csv_file:
    df_data = pd.read_csv(csv_file)
df_data.head()

,ImageId,PredictionString,origin_size
0,ILSVRC2012_val_00048981,n03995372 85 1 499 272,"(500, 360)"
1,ILSVRC2012_val_00037956,n03481172 131 0 499 254,"(500, 333)"
2,ILSVRC2012_val_00026161,n02108000 38 0 464 280,"(500, 334)"
3,ILSVRC2012_val_00026171,n03109150 0 14 216 299,"(225, 300)"
4,ILSVRC2012_val_00008726,n02119789 255 142 454 329 n02119789 44 21 322 ...,"(500, 357)"


In [8]:
get_boxes(df_data, 'ILSVRC2012_val_00026171')

[{'xmin': 0, 'ymin': 14, 'xmax': 216, 'ymax': 299}]

In [9]:
def clean_list(list_dir, patterns=['.DS_Store', '.ipynb_checkpoints']):
    for pattern in patterns:
        if pattern in list_dir: list_dir.remove(pattern)
    return list_dir

## cropping images

In [10]:
from PIL import Image 

def square_box(xmin, ymin, xmax, ymax):
    temp = ((xmax-xmin)-(ymax-ymin))//2 # signed radius
    if temp > 0 :
        ymin -= temp
        ymax += temp
    else:
        xmin += temp
        xmax -= temp
    return xmin, ymin, xmax, ymax


for folder in args.folders :
    # first level
    print(f'\nFolder \"{folder}\"')
    source_folder = os.path.join(args.root, folder)
    boxes_folder = os.path.join(args.target_root, folder)
    os.makedirs(boxes_folder, exist_ok=True)

    # second level
    with open(f'data/LOC_{folder}_solution.csv', 'r') as csv_file:
        df_data = pd.read_csv(csv_file)
    for i_label, label_id in enumerate(revlabels_dico.keys()):
        print(f'Scraping images for id \"{label_id}\" : {labels[i_label]} ', end='')
        img_source_folder = os.path.join(source_folder, label_id)
        target_folder = os.path.join(boxes_folder, label_id)
        if True: #not os.path.isdir(target_folder):
            os.makedirs(target_folder, exist_ok=True)
            print(os.listdir(img_source_folder))
            for imgs in  clean_list(os.listdir(img_source_folder)):
                data_local = os.path.join(img_source_folder, imgs)
                objects = get_boxes(df_data, imgs.split('.')[0])

                original_image = Image.open(data_local, mode='r').convert('RGB')
                for i_obj, object in enumerate(objects): #if len(obj)  > 0 :
                    xmin = object['xmin']
                    ymin = object['ymin']
                    xmax = object['xmax']
                    ymax = object['ymax']

                    crop_image = original_image.crop(square_box(xmin, ymin, xmax, ymax))
                    no = '' if i_obj==0 else f'_{i_obj}'
                    img_name = imgs.split('.')[0] + no + '.jpg'
                    crop_image.save(os.path.join(target_folder, img_name))

        print(f' - in:  {len(clean_list(os.listdir(img_source_folder)))} / out:  {len(clean_list(os.listdir(target_folder)))}')


Folder "val"
Scraping images for id "n01440764" : tench []
 - in:  0 / out:  0
Scraping images for id "n01443537" : goldfish []
 - in:  0 / out:  0
Scraping images for id "n01484850" : great_white_shark []
 - in:  0 / out:  0
Scraping images for id "n01491361" : tiger_shark []
 - in:  0 / out:  0
Scraping images for id "n01494475" : hammerhead []
 - in:  0 / out:  0
Scraping images for id "n01496331" : electric_ray []
 - in:  0 / out:  0
Scraping images for id "n01498041" : stingray []
 - in:  0 / out:  0
Scraping images for id "n01514668" : cock []
 - in:  0 / out:  0
Scraping images for id "n01514859" : hen []
 - in:  0 / out:  0
Scraping images for id "n01518878" : ostrich []
 - in:  0 / out:  0
Scraping images for id "n01530575" : brambling []
 - in:  0 / out:  0
Scraping images for id "n01531178" : goldfinch []
 - in:  0 / out:  0
Scraping images for id "n01532829" : house_finch []
 - in:  0 / out:  0
Scraping images for id "n01534433" : junco []
 - in:  0 / out:  0
Scraping imag

 - in:  0 / out:  0
Scraping images for id "n01695060" : komodo_dragon []
 - in:  0 / out:  0
Scraping images for id "n01697457" : african_crocodile []
 - in:  0 / out:  0
Scraping images for id "n01698640" : american_alligator []
 - in:  0 / out:  0
Scraping images for id "n01704323" : triceratops []
 - in:  0 / out:  0
Scraping images for id "n01728572" : thunder_snake []
 - in:  0 / out:  0
Scraping images for id "n01728920" : ringneck_snake []
 - in:  0 / out:  0
Scraping images for id "n01729322" : hognose_snake []
 - in:  0 / out:  0
Scraping images for id "n01729977" : green_snake []
 - in:  0 / out:  0
Scraping images for id "n01734418" : king_snake []
 - in:  0 / out:  0
Scraping images for id "n01735189" : garter_snake []
 - in:  0 / out:  0
Scraping images for id "n01737021" : water_snake []
 - in:  0 / out:  0
Scraping images for id "n01739381" : vine_snake []
 - in:  0 / out:  0
Scraping images for id "n01740131" : night_snake []
 - in:  0 / out:  0
Scraping images for id 

[]
 - in:  0 / out:  0
Scraping images for id "n02002724" : black_stork []
 - in:  0 / out:  0
Scraping images for id "n02006656" : spoonbill []
 - in:  0 / out:  0
Scraping images for id "n02007558" : flamingo []
 - in:  0 / out:  0
Scraping images for id "n02009229" : little_blue_heron []
 - in:  0 / out:  0
Scraping images for id "n02009912" : american_egret []
 - in:  0 / out:  0
Scraping images for id "n02011460" : bittern []
 - in:  0 / out:  0
Scraping images for id "n02012849" : crane []
 - in:  0 / out:  0
Scraping images for id "n02013706" : limpkin []
 - in:  0 / out:  0
Scraping images for id "n02017213" : european_gallinule []
 - in:  0 / out:  0
Scraping images for id "n02018207" : american_coot []
 - in:  0 / out:  0
Scraping images for id "n02018795" : bustard []
 - in:  0 / out:  0
Scraping images for id "n02025239" : ruddy_turnstone []
 - in:  0 / out:  0
Scraping images for id "n02027492" : red-backed_sandpiper []
 - in:  0 / out:  0
Scraping images for id "n02028035

 - in:  0 / out:  0
Scraping images for id "n02099849" : chesapeake_bay_retriever []
 - in:  0 / out:  0
Scraping images for id "n02100236" : german_short-haired_pointer []
 - in:  0 / out:  0
Scraping images for id "n02100583" : vizsla []
 - in:  0 / out:  0
Scraping images for id "n02100735" : english_setter []
 - in:  0 / out:  0
Scraping images for id "n02100877" : irish_setter []
 - in:  0 / out:  0
Scraping images for id "n02101006" : gordon_setter []
 - in:  0 / out:  0
Scraping images for id "n02101388" : brittany_spaniel []
 - in:  0 / out:  0
Scraping images for id "n02101556" : clumber []
 - in:  0 / out:  0
Scraping images for id "n02102040" : english_springer []
 - in:  0 / out:  0
Scraping images for id "n02102177" : welsh_springer_spaniel []
 - in:  0 / out:  0
Scraping images for id "n02102318" : cocker_spaniel []
 - in:  0 / out:  0
Scraping images for id "n02102480" : sussex_spaniel []
 - in:  0 / out:  0
Scraping images for id "n02102973" : irish_water_spaniel []
 - 

 - in:  0 / out:  0
Scraping images for id "n02127052" : lynx []
 - in:  0 / out:  0
Scraping images for id "n02128385" : leopard []
 - in:  0 / out:  0
Scraping images for id "n02128757" : snow_leopard []
 - in:  0 / out:  0
Scraping images for id "n02128925" : jaguar []
 - in:  0 / out:  0
Scraping images for id "n02129165" : lion []
 - in:  0 / out:  0
Scraping images for id "n02129604" : tiger []
 - in:  0 / out:  0
Scraping images for id "n02130308" : cheetah []
 - in:  0 / out:  0
Scraping images for id "n02132136" : brown_bear []
 - in:  0 / out:  0
Scraping images for id "n02133161" : american_black_bear []
 - in:  0 / out:  0
Scraping images for id "n02134084" : ice_bear []
 - in:  0 / out:  0
Scraping images for id "n02134418" : sloth_bear []
 - in:  0 / out:  0
Scraping images for id "n02137549" : mongoose []
 - in:  0 / out:  0
Scraping images for id "n02138441" : meerkat []
 - in:  0 / out:  0
Scraping images for id "n02165105" : tiger_beetle []
 - in:  0 / out:  0
Scrapin

...

[]
 - in:  0 / out:  0
Scraping images for id "n03733805" : measuring_cup []
 - in:  0 / out:  0
Scraping images for id "n03742115" : medicine_chest []
 - in:  0 / out:  0
Scraping images for id "n03743016" : megalith []
 - in:  0 / out:  0
Scraping images for id "n03759954" : microphone []
 - in:  0 / out:  0
Scraping images for id "n03761084" : microwave []
 - in:  0 / out:  0
Scraping images for id "n03763968" : military_uniform []
 - in:  0 / out:  0
Scraping images for id "n03764736" : milk_can []
 - in:  0 / out:  0
Scraping images for id "n03769881" : minibus []
 - in:  0 / out:  0
Scraping images for id "n03770439" : miniskirt []
 - in:  0 / out:  0
Scraping images for id "n03770679" : minivan []
 - in:  0 / out:  0
Scraping images for id "n03773504" : missile []
 - in:  0 / out:  0
Scraping images for id "n03775071" : mitten []
 - in:  0 / out:  0
Scraping images for id "n03775546" : mixing_bowl []
 - in:  0 / out:  0
Scraping images for id "n03776460" : mobile_home []
 - in: 

[]
 - in:  0 / out:  0
Scraping images for id "n03976467" : polaroid_camera []
 - in:  0 / out:  0
Scraping images for id "n03976657" : pole []
 - in:  0 / out:  0
Scraping images for id "n03977966" : police_van []
 - in:  0 / out:  0
Scraping images for id "n03980874" : poncho []
 - in:  0 / out:  0
Scraping images for id "n03982430" : pool_table []
 - in:  0 / out:  0
Scraping images for id "n03983396" : pop_bottle []
 - in:  0 / out:  0
Scraping images for id "n03991062" : pot []
 - in:  0 / out:  0
Scraping images for id "n03992509" : potter's_wheel []
 - in:  0 / out:  0
Scraping images for id "n03995372" : power_drill []
 - in:  0 / out:  0
Scraping images for id "n03998194" : prayer_rug []
 - in:  0 / out:  0
Scraping images for id "n04004767" : printer []
 - in:  0 / out:  0
Scraping images for id "n04005630" : prison []
 - in:  0 / out:  0
Scraping images for id "n04008634" : projectile []
 - in:  0 / out:  0
Scraping images for id "n04009552" : projector []
 - in:  0 / out:  

[]
 - in:  0 / out:  0
Scraping images for id "n04252077" : snowmobile []
 - in:  0 / out:  0
Scraping images for id "n04252225" : snowplow []
 - in:  0 / out:  0
Scraping images for id "n04254120" : soap_dispenser []
 - in:  0 / out:  0
Scraping images for id "n04254680" : soccer_ball []
 - in:  0 / out:  0
Scraping images for id "n04254777" : sock []
 - in:  0 / out:  0
Scraping images for id "n04258138" : solar_dish []
 - in:  0 / out:  0
Scraping images for id "n04259630" : sombrero []
 - in:  0 / out:  0
Scraping images for id "n04263257" : soup_bowl []
 - in:  0 / out:  0
Scraping images for id "n04264628" : space_bar []
 - in:  0 / out:  0
Scraping images for id "n04265275" : space_heater []
 - in:  0 / out:  0
Scraping images for id "n04266014" : space_shuttle []
 - in:  0 / out:  0
Scraping images for id "n04270147" : spatula []
 - in:  0 / out:  0
Scraping images for id "n04273569" : speedboat []
 - in:  0 / out:  0
Scraping images for id "n04275548" : spider_web []
 - in:  0

 - in:  0 / out:  0
Scraping images for id "n04523525" : vault []
 - in:  0 / out:  0
Scraping images for id "n04525038" : velvet []
 - in:  0 / out:  0
Scraping images for id "n04525305" : vending_machine []
 - in:  0 / out:  0
Scraping images for id "n04532106" : vestment []
 - in:  0 / out:  0
Scraping images for id "n04532670" : viaduct []
 - in:  0 / out:  0
Scraping images for id "n04536866" : violin []
 - in:  0 / out:  0
Scraping images for id "n04540053" : volleyball []
 - in:  0 / out:  0
Scraping images for id "n04542943" : waffle_iron []
 - in:  0 / out:  0
Scraping images for id "n04548280" : wall_clock []
 - in:  0 / out:  0
Scraping images for id "n04548362" : wallet []
 - in:  0 / out:  0
Scraping images for id "n04550184" : wardrobe []
 - in:  0 / out:  0
Scraping images for id "n04552348" : warplane []
 - in:  0 / out:  0
Scraping images for id "n04553703" : washbasin []
 - in:  0 / out:  0
Scraping images for id "n04554684" : washer []
 - in:  0 / out:  0
Scraping im

[]
 - in:  0 / out:  0
Scraping images for id "n07920052" : espresso []
 - in:  0 / out:  0
Scraping images for id "n07930864" : cup []
 - in:  0 / out:  0
Scraping images for id "n07932039" : eggnog []
 - in:  0 / out:  0
Scraping images for id "n09193705" : alp []
 - in:  0 / out:  0
Scraping images for id "n09229709" : bubble []
 - in:  0 / out:  0
Scraping images for id "n09246464" : cliff []
 - in:  0 / out:  0
Scraping images for id "n09256479" : coral_reef []
 - in:  0 / out:  0
Scraping images for id "n09288635" : geyser []
 - in:  0 / out:  0
Scraping images for id "n09332890" : lakeside []
 - in:  0 / out:  0
Scraping images for id "n09399592" : promontory []
 - in:  0 / out:  0
Scraping images for id "n09421951" : sandbar []
 - in:  0 / out:  0
Scraping images for id "n09428293" : seashore []
 - in:  0 / out:  0
Scraping images for id "n09468604" : valley []
 - in:  0 / out:  0
Scraping images for id "n09472597" : volcano []
 - in:  0 / out:  0
Scraping images for id "n09835